# 02 — Dataset EDA and Labels

Inspect metadata schema, split sizes, label definitions, class imbalance, and duration values without loading media files.

> **Project:** Multimodal Deepfake Detection using LAV-DF (audio + video). Run cells in order. Confirm dataset paths and split definitions before training. Never fit preprocessing or tune thresholds on the test split.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Update this path to the CSV generated by scripts/01_explore_lavdf.py.
AUDIT_CSV = Path("../../results/eda/metadata_inventory.csv")
if AUDIT_CSV.exists():
    df = pd.read_csv(AUDIT_CSV)
    display(df.head())
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())
else:
    print(f"Audit CSV not found: {AUDIT_CSV.resolve()}")
    print("Run the environment audit / repository script first.")

In [ ]:
if "df" in globals():
    for col in ["split", "binary_label", "label_4class"]:
        if col in df.columns:
            print(f"\n{col} counts:")
            display(df[col].value_counts(dropna=False).rename_axis(col).to_frame("count"))

In [ ]:
if "df" in globals() and "split" in df.columns and "binary_label" in df.columns:
    display(pd.crosstab(df["split"], df["binary_label"], margins=True))
    ax = pd.crosstab(df["split"], df["binary_label"]).plot(kind="bar")
    ax.set_title("Binary label counts by split")
    ax.set_ylabel("Records")
    plt.tight_layout()
    plt.show()

## Label policy
Binary task: `real = 0`, `fake = 1` when metadata indicates at least one fake segment. Four-class task: real, video-only manipulation, audio-only manipulation, and both modalities manipulated. Validate the source metadata semantics before treating these as ground truth.

In [ ]:
if "df" in globals():
    print("Missing values per column:")
    display(df.isna().sum().sort_values(ascending=False).to_frame("missing"))